## 0. Setup & Config

In [1]:
import importlib
import json
import subprocess
import sys
from pathlib import Path


def _import_or_install(import_name: str, package_name: str | None = None):
    try:
        return importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name or import_name])
        return importlib.import_module(import_name)


np = _import_or_install("numpy")
pd = _import_or_install("pandas")
sklearn = _import_or_install("sklearn", "scikit-learn")
ce = _import_or_install("category_encoders", "category-encoders")
cb = _import_or_install("catboost")
lgb = _import_or_install("lightgbm")
matplotlib = _import_or_install("matplotlib")
plt = importlib.import_module("matplotlib.pyplot")
optuna = _import_or_install("optuna")
sns = _import_or_install("seaborn")
xgb = _import_or_install("xgboost")

/usr/local/python/3.14.2/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
CHARTS_DIR = OUTPUT_DIR / 'charts'
MODELS_DIR = OUTPUT_DIR / 'models'
RAW_TRAIN_PATH = DATA_DIR / 'train.csv'
RAW_TEST_PATH = DATA_DIR / 'test.csv'
SAMPLE_SUBMISSION_PATH = DATA_DIR / 'sample_submission.csv'



In [3]:
train = pd.read_csv(RAW_TRAIN_PATH)
test = pd.read_csv(RAW_TEST_PATH)

print(train.shape)
print(test.shape)

(6818, 13)
(1705, 12)


In [4]:
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 1. EDA & Structural Diagnostics

In [5]:
PRODUCT_KEY = 'product_code'
STORE_KEY = 'store_code'
CATEGORY_KEY = 'product_category'
TARGET_KEY = 'total_sales'

In [6]:
missing_cols = ['product_code', 'store_code', 'product_category', 'fat_content', 'store_format', 'store_location_tier', 'store_size']

print('Train missingness')
print(train.isnull().sum())
print()
print('Test missingness')
print(test.isnull().sum())
print()

for name, frame in [('train', train), ('test', test)]:
    print(f'{name} cardinality')
    print(frame[missing_cols].nunique())
    print()

Train missingness
id                        0
product_code              0
product_weight_kg      1225
fat_content               0
shelf_visibility          0
product_category          0
product_price             0
store_code                0
store_age_years           0
store_size             1919
store_location_tier       0
store_format              0
total_sales               0
dtype: int64

Test missingness
id                       0
product_code             0
product_weight_kg      306
fat_content              0
shelf_visibility         0
product_category         0
product_price            0
store_code               0
store_age_years          0
store_size             491
store_location_tier      0
store_format             0
dtype: int64

train cardinality
product_code           1555
store_code               10
product_category         48
fat_content               2
store_format              4
store_location_tier       3
store_size                3
dtype: int64

test cardinality
prod

In [7]:
store_overlap = set(test[STORE_KEY]) <= set(train[STORE_KEY])
store_cold_start_count = len(set(test[STORE_KEY]) - set(train[STORE_KEY]))
print('store_overlap:', store_overlap)
print('store_cold_start_count:', store_cold_start_count)

store_overlap: True
store_cold_start_count: 0


In [8]:
product_overlap = set(test[PRODUCT_KEY]) <= set(train[PRODUCT_KEY])
product_cold_start_count = len(set(test[PRODUCT_KEY]) - set(train[PRODUCT_KEY]))
print('product_overlap:', product_overlap)
print('product_cold_start_count:', product_cold_start_count)

product_overlap: False
product_cold_start_count: 4


In [9]:
pair_counts = train.groupby([PRODUCT_KEY, STORE_KEY]).size()
print(pair_counts.describe())
print('max_pair_count:', pair_counts.max())
max_pair_count = pair_counts.max()

count    6818.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
dtype: float64
max_pair_count: 1


In [10]:
store_size_summary = train.groupby(STORE_KEY)['store_size'].agg(
    nunique_non_null=lambda s: s.dropna().nunique(),
    missing_rows=lambda s: s.isna().sum(),
)
stores_missing_size = store_size_summary[store_size_summary['missing_rows'] > 0].index.tolist()
print(store_size_summary)
print('stores_missing_size:', stores_missing_size)

            nunique_non_null  missing_rows
store_code                                
STORE-7WS                  1             0
STORE-89Z                  1             0
STORE-9RG                  1             0
STORE-AGY                  1             0
STORE-DKU                  0           736
STORE-HL7                  1             0
STORE-JOR                  0           439
STORE-OYG                  0           744
STORE-T5G                  1             0
STORE-YLW                  1             0
stores_missing_size: ['STORE-DKU', 'STORE-JOR', 'STORE-OYG']


In [11]:
all_sales = pd.concat([
    train.drop(columns=[TARGET_KEY]),
    test,
], ignore_index=True)

weight_summary = all_sales.groupby(PRODUCT_KEY)['product_weight_kg'].agg(
    missing_rows=lambda s: s.isna().sum(),
    total_rows='size',
)
products_fully_missing_weight = weight_summary.index[weight_summary['missing_rows'] == weight_summary['total_rows']].tolist()
print(weight_summary.head())
print('products_fully_missing_weight_count:', len(products_fully_missing_weight))
print('products_fully_missing_weight:', products_fully_missing_weight)

              missing_rows  total_rows
product_code                          
PRD-00CVW9               0           5
PRD-00EIUE               2           6
PRD-00V830               1           2
PRD-02NFGI               3           7
PRD-031L36               2           6
products_fully_missing_weight_count: 4
products_fully_missing_weight: ['PRD-7JH2LK', 'PRD-DH3TKZ', 'PRD-E6K088', 'PRD-GSRI2A']


In [12]:
train['product_category_normalized'] = train[CATEGORY_KEY].str.lower().str.strip()
test['product_category_normalized'] = test[CATEGORY_KEY].str.lower().str.strip()

raw_nunique = train[CATEGORY_KEY].nunique()
normalized_nunique = train['product_category_normalized'].nunique()
product_categories_normalized = sorted(train['product_category_normalized'].dropna().unique().tolist())

print('raw_nunique:', raw_nunique)
print('normalized_nunique:', normalized_nunique)
print('product_categories_normalized:', product_categories_normalized)

raw_nunique: 48
normalized_nunique: 16
product_categories_normalized: ['baking goods', 'breads', 'breakfast', 'canned', 'dairy', 'frozen foods', 'fruits and vegetables', 'hard drinks', 'health and hygiene', 'household', 'meat', 'others', 'seafood', 'snack foods', 'soft drinks', 'starchy foods']


In [13]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

CHARTS_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train[TARGET_KEY], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Total Sales')
sns.histplot(np.log1p(train[TARGET_KEY]), kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('log1p(Total Sales)')
fig.tight_layout()
fig.savefig(CHARTS_DIR / 'target_distribution.png', dpi=150, bbox_inches='tight')
plt.close(fig)

print('sales_skew:', train[TARGET_KEY].skew())
print('log1p_sales_skew:', np.log1p(train[TARGET_KEY]).skew())

sales_skew: 1.1539947519138116
log1p_sales_skew: -0.8951280115081985


In [14]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.barplot(data=train, x='store_format', y=TARGET_KEY, estimator=np.mean, errorbar=None, ax=axes[0, 0], color='teal')
axes[0, 0].set_title('Mean Total Sales by Store Format')
axes[0, 0].tick_params(axis='x', rotation=20)

sns.boxplot(data=train, x='store_format', y=TARGET_KEY, ax=axes[0, 1], color='lightgray')
axes[0, 1].set_title('Total Sales by Store Format')
axes[0, 1].tick_params(axis='x', rotation=20)

sns.barplot(data=train, x='store_location_tier', y=TARGET_KEY, estimator=np.mean, errorbar=None, ax=axes[1, 0], color='slateblue')
axes[1, 0].set_title('Mean Total Sales by Store Location Tier')
axes[1, 0].tick_params(axis='x', rotation=20)

sns.barplot(data=train, x='product_category_normalized', y=TARGET_KEY, estimator=np.mean, errorbar=None, ax=axes[1, 1], color='coral')
axes[1, 1].set_title('Mean Total Sales by Normalized Product Category')
axes[1, 1].tick_params(axis='x', rotation=45)

fig.tight_layout()
fig.savefig(CHARTS_DIR / 'sales_by_factors.png', dpi=150, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=train, x='store_format', y=TARGET_KEY, estimator=np.mean, errorbar=None, ax=ax, color='teal')
ax.set_title('Mean Total Sales by Store Format')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(CHARTS_DIR / 'mean_sales_by_store_format.png', dpi=150, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=train, x='store_format', y=TARGET_KEY, ax=ax, color='lightgray')
ax.set_title('Total Sales by Store Format')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(CHARTS_DIR / 'boxplot_sales_by_store_format.png', dpi=150, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=train, x='store_location_tier', y=TARGET_KEY, estimator=np.mean, errorbar=None, ax=ax, color='slateblue')
ax.set_title('Mean Total Sales by Store Location Tier')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(CHARTS_DIR / 'mean_sales_by_store_location_tier.png', dpi=150, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(16, 6))
sns.barplot(data=train, x='product_category_normalized', y=TARGET_KEY, estimator=np.mean, errorbar=None, ax=ax, color='coral')
ax.set_title('Mean Total Sales by Normalized Product Category')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(CHARTS_DIR / 'mean_sales_by_product_category.png', dpi=150, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 6))
store_size_summary['missing_rows'].plot(kind='bar', ax=ax, color='firebrick')
ax.set_title('Missing Store Size Rows by Store')
ax.set_ylabel('Missing rows')
fig.tight_layout()
fig.savefig(CHARTS_DIR / 'store_size_missingness_by_store.png', dpi=150, bbox_inches='tight')
plt.close(fig)

Section summary: the diagnostics section computes the cold-start counts, pair multiplicity, store-size missingness, fully missing product-weight list, normalized product categories, target skew, and chart files needed for Tasks 2 and 3. The main modeling implication is to treat store overlap as safe, treat product overlap according to the observed count, and use the store-size and weight summaries to decide whether imputation can be store-level or product-level rather than row-level.

## 2. Cleaning

In [15]:
def clean_data(train: pd.DataFrame, test: pd.DataFrame, stores_missing_size: list,
               products_fully_missing_weight: list) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns (train_clean, test_clean), each with:
      - product_category: lowercased/stripped
      - product_weight_kg: imputed, + product_weight_missing flag (0/1)
      - store_size: imputed, + store_size_missing flag (0/1)
    Does not drop or reorder rows. Does not touch total_sales.
    """
    train_clean = train.copy()
    test_clean = test.copy()

    for frame in (train_clean, test_clean):
        frame['product_category'] = frame['product_category'].str.lower().str.strip()

    combined = pd.concat(
        [
            train_clean.drop(columns=[TARGET_KEY]),
            test_clean,
        ],
        ignore_index=True,
    )

    category_means = combined.groupby('product_category')['product_weight_kg'].mean()
    product_means = combined.groupby(PRODUCT_KEY)['product_weight_kg'].mean()
    category_medians = combined.groupby('product_category')['product_weight_kg'].median()

    for frame in (train_clean, test_clean):
        frame['product_weight_missing'] = frame['product_weight_kg'].isna().astype(int)
        frame['product_weight_kg'] = frame['product_weight_kg'].fillna(frame[PRODUCT_KEY].map(product_means))
        if products_fully_missing_weight:
            mask = frame['product_weight_kg'].isna() & frame[PRODUCT_KEY].isin(products_fully_missing_weight)
            frame.loc[mask, 'product_weight_kg'] = frame.loc[mask, 'product_category'].map(category_means)
            frame['product_weight_kg'] = frame['product_weight_kg'].fillna(frame['product_category'].map(category_medians))

    store_size_lookup = (
        train_clean.loc[~train_clean['store_size'].isna(), [STORE_KEY, 'store_format', 'store_location_tier', 'store_size']]
        .groupby(['store_format', 'store_location_tier'])['store_size']
        .agg(lambda s: s.mode().iloc[0])
    )
    global_store_size_mode = train_clean.loc[~train_clean['store_size'].isna(), 'store_size'].mode().iloc[0]

    for frame in (train_clean, test_clean):
        frame['store_size_missing'] = frame['store_size'].isna().astype(int)
        imputed_store_size = (
            frame[['store_format', 'store_location_tier']]
            .apply(lambda row: store_size_lookup.get((row['store_format'], row['store_location_tier']), global_store_size_mode), axis=1)
        )
        frame['store_size'] = frame['store_size'].fillna(imputed_store_size)
        frame['store_size'] = frame['store_size'].fillna(global_store_size_mode)

    assert sorted(train_clean['product_category'].dropna().unique().tolist()) == product_categories_normalized
    assert sorted(test_clean['product_category'].dropna().unique().tolist()) == product_categories_normalized

    return train_clean, test_clean

In [16]:
train_clean, test_clean = clean_data(train, test, stores_missing_size, products_fully_missing_weight)
print(train_clean.isnull().sum())
print()
print(test_clean.isnull().sum())
print()
print(train_clean.shape)
print(test_clean.shape)

id                             0
product_code                   0
product_weight_kg              0
fat_content                    0
shelf_visibility               0
product_category               0
product_price                  0
store_code                     0
store_age_years                0
store_size                     0
store_location_tier            0
store_format                   0
total_sales                    0
product_category_normalized    0
product_weight_missing         0
store_size_missing             0
dtype: int64

id                             0
product_code                   0
product_weight_kg              0
fat_content                    0
shelf_visibility               0
product_category               0
product_price                  0
store_code                     0
store_age_years                0
store_size                     0
store_location_tier            0
store_format                   0
product_category_normalized    0
product_weight_missing       

Task 2 branch note: Task 1 showed `store_size_summary['nunique_non_null']` is 1 for every store that has any known `store_size`, so Section 2 uses the store-level branch. It builds the imputation map at runtime from known `(store_format, store_location_tier) -> most common store_size` pairs, then fills the stores listed in `stores_missing_size` with a global fallback only if a combination has no observed example. That choice is justified by the store-level constancy finding in Task 1 and keeps the logic data-driven rather than hardcoded.

In [17]:
partial_weight_product = next(
    product for product, group in train.groupby(PRODUCT_KEY)
    if group['product_weight_kg'].isna().any() and group['product_weight_kg'].notna().any()
)
raw_mean_weight = train.loc[train[PRODUCT_KEY] == partial_weight_product, 'product_weight_kg'].mean()
imputed_mean_weight = train_clean.loc[train_clean[PRODUCT_KEY] == partial_weight_product, 'product_weight_kg'].dropna().iloc[0]
print('partial_weight_product:', partial_weight_product)
print('raw_mean_weight:', raw_mean_weight)
print('imputed_mean_weight:', imputed_mean_weight)
print('match:', np.isclose(raw_mean_weight, imputed_mean_weight))

partial_weight_product: PRD-00V830
raw_mean_weight: 9.679
imputed_mean_weight: 9.679
match: True


In [18]:
assert train_clean.drop(columns=[TARGET_KEY]).isnull().sum().sum() == 0
assert test_clean.isnull().sum().sum() == 0
assert len(train_clean) == len(train)
assert len(test_clean) == len(test)
print('cleaning checks passed')

cleaning checks passed


## 3. Feature Engineering

In [19]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

def add_target_agg_feature(
    train: pd.DataFrame,
    test: pd.DataFrame,
    group_cols: list[str],
    target_col: str = 'total_sales',
    agg: str = 'mean',
    n_folds: int = 5,
    new_col_name: str | None = None,
    smoothing: float | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns (train_with_feature, test_with_feature).
    train: feature generated OUT-OF-FOLD using sklearn KFold(n_folds, shuffle=True, random_state=42).
    test: feature generated once using the FULL train set.
    Missing groups filled with the global mean of target_col.
    """
    train_with_feature = train.copy()
    test_with_feature = test.copy()
    feature_name = new_col_name or f"{'_'.join(group_cols)}_{agg}_sales"
    global_mean = train[target_col].mean()

    kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    oof_values = pd.Series(index=train.index, dtype=float)

    for train_idx, val_idx in kfold.split(train_with_feature):
        fold_train = train_with_feature.iloc[train_idx]
        if agg == 'count':
            fold_stats = fold_train.groupby(group_cols)[target_col].count()
        else:
            grouped = fold_train.groupby(group_cols)[target_col]
            if agg == 'mean':
                fold_stats = grouped.mean()
                if smoothing is not None:
                    sums = grouped.sum()
                    counts = grouped.count()
                    fold_stats = (sums + smoothing * global_mean) / (counts + smoothing)
            elif agg == 'median':
                fold_stats = grouped.median()
            elif agg == 'std':
                fold_stats = grouped.std()
            else:
                raise ValueError(f'Unsupported agg: {agg}')

        mapped = train_with_feature.iloc[val_idx][group_cols].merge(
            fold_stats.rename(feature_name).reset_index(),
            on=group_cols,
            how='left',
        )[feature_name]
        oof_values.iloc[val_idx] = mapped.fillna(global_mean).to_numpy()

    train_with_feature[feature_name] = oof_values.fillna(global_mean)

    full_grouped = train_with_feature.groupby(group_cols)[target_col]
    if agg == 'count':
        full_stats = full_grouped.count()
    elif agg == 'mean':
        full_stats = full_grouped.mean()
        if smoothing is not None:
            sums = full_grouped.sum()
            counts = full_grouped.count()
            full_stats = (sums + smoothing * global_mean) / (counts + smoothing)
    elif agg == 'median':
        full_stats = full_grouped.median()
    elif agg == 'std':
        full_stats = full_grouped.std()
    else:
        raise ValueError(f'Unsupported agg: {agg}')

    train_with_feature[feature_name] = train_with_feature[feature_name].fillna(global_mean)
    test_with_feature[feature_name] = (
        test_with_feature[group_cols].merge(
            full_stats.rename(feature_name).reset_index(),
            on=group_cols,
            how='left',
        )[feature_name].fillna(global_mean).to_numpy()
    )

    return train_with_feature, test_with_feature

In [20]:
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds: price_per_kg, visibility_price_interaction, visibility_weight_interaction,
    store_age_squared, log_price, log_weight, and ordinal encodings for store_size,
    store_location_tier, store_format.
    Pure row-wise transforms — safe to apply identically to train and test.
    """
    df = df.copy()

    store_size_order = {'Small': 1, 'Medium': 2, 'Large': 3}
    tier_order = {'Tier_1': 1, 'Tier_2': 2, 'Tier_3': 3}
    format_order = {
        'Corner Shop': 1,
        'Standard Supermarket': 2,
        'Superstore': 3,
        'Flagship Hypermarket': 4,
    }

    df['price_per_kg'] = df['product_price'] / df['product_weight_kg']
    df['visibility_price_interaction'] = df['shelf_visibility'] * df['product_price']
    df['visibility_weight_interaction'] = df['shelf_visibility'] * df['product_weight_kg']
    df['store_age_squared'] = df['store_age_years'] ** 2
    df['log_price'] = np.log1p(df['product_price'])
    df['log_weight'] = np.log1p(df['product_weight_kg'])
    df['store_size_ord'] = df['store_size'].map(store_size_order).astype(float)
    df['store_location_tier_ord'] = df['store_location_tier'].map(tier_order).astype(float)
    df['store_format_ord'] = df['store_format'].map(format_order).astype(float)

    return df

Task 3 branch note: `max_pair_count` was 1 in Task 1, so the product-store interaction is not built as a raw `prod_store_mean_sales` feature. A raw pair mean would be a one-row lookup of the target itself, so the notebook stays with product-level, store-level, and lower-cardinality interaction aggregates only.

In [21]:
train_feat = add_derived_features(train_clean)
test_feat = add_derived_features(test_clean)

agg_specs = [
    (['product_code'], 'mean', 'prod_mean_sales', None),
    (['product_code'], 'median', 'prod_median_sales', None),
    (['product_code'], 'std', 'prod_std_sales', None),
    (['product_code'], 'count', 'prod_count', None),
    (['store_code'], 'mean', 'store_mean_sales', None),
    (['store_code'], 'std', 'store_std_sales', None),
    (['store_format'], 'mean', 'format_mean_sales', None),
    (['store_location_tier'], 'mean', 'tier_mean_sales', None),
    (['store_size'], 'mean', 'size_mean_sales', None),
    (['product_code', 'store_format'], 'mean', 'prod_format_mean_sales', 10),
    (['product_code', 'store_location_tier'], 'mean', 'prod_tier_mean_sales', 10),
]

for group_cols, agg, feature_name, smoothing in agg_specs:
    train_feat, test_feat = add_target_agg_feature(
        train_feat,
        test_feat,
        group_cols=group_cols,
        target_col=TARGET_KEY,
        agg=agg,
        n_folds=5,
        new_col_name=feature_name,
        smoothing=smoothing,
    )

target_derived_features = [spec[2] for spec in agg_specs]
print(train_feat[target_derived_features].head())
print(test_feat[target_derived_features].head())

   prod_mean_sales  prod_median_sales  prod_std_sales  prod_count  \
0      1355.000000           1324.730      574.803085         3.0   
1       935.425000            935.425      146.958002         2.0   
2       387.682500            172.225      522.856584         4.0   
3      4538.960000           4538.960     2174.756574         1.0   
4      1937.826667           2613.600     1192.625551         3.0   

   store_mean_sales  store_std_sales  format_mean_sales  tier_mean_sales  \
0       2241.792659      1484.255547        2330.516069      2248.711299   
1       2249.251510      1484.567224        2327.119721      1886.769480   
2       2459.615639      1523.161535        2327.119721      1886.769480   
3       3642.204269      2057.442661        3642.204269      2247.659776   
4       2459.996444      1515.374087        2313.188090      2338.564123   

   size_mean_sales  prod_format_mean_sales  prod_tier_mean_sales  
0      2241.792659             1989.018811           2153.809

In [22]:
leakage_corrs = train_feat[target_derived_features + [TARGET_KEY]].corr(numeric_only=True)[TARGET_KEY].drop(TARGET_KEY).sort_values(ascending=False)
print(leakage_corrs)
print('flagged_features:', leakage_corrs[leakage_corrs >= 0.9].index.tolist())

store_mean_sales          0.488779
format_mean_sales         0.488148
store_std_sales           0.481295
prod_mean_sales           0.398740
prod_format_mean_sales    0.398384
prod_median_sales         0.384008
prod_std_sales            0.342491
prod_tier_mean_sales      0.207022
tier_mean_sales           0.111721
size_mean_sales           0.055990
prod_count                0.020689
Name: total_sales, dtype: float64
flagged_features: []


In [23]:
spot_idx = train_clean.index[0]
spot_product = train_clean.loc[spot_idx, PRODUCT_KEY]
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold_lookup = {}
for fold_id, (_, val_idx) in enumerate(kfold.split(train_clean)):
    for idx in val_idx:
        fold_lookup[idx] = fold_id
spot_fold = fold_lookup[spot_idx]
fold_train_idx = [idx for idx, fold_id in fold_lookup.items() if fold_id != spot_fold]
fold_train = train_clean.loc[fold_train_idx]
recomputed_prod_mean = fold_train.loc[fold_train[PRODUCT_KEY] == spot_product, TARGET_KEY].mean()
assigned_prod_mean = train_feat.loc[spot_idx, 'prod_mean_sales']
print('spot_idx:', spot_idx)
print('spot_product:', spot_product)
print('recomputed_prod_mean:', recomputed_prod_mean)
print('assigned_prod_mean:', assigned_prod_mean)
print('match:', np.isclose(recomputed_prod_mean, assigned_prod_mean))

spot_idx: 0
spot_product: PRD-PRFP9S
recomputed_prod_mean: 1355.0
assigned_prod_mean: 1355.0
match: True


In [25]:
def train_baseline(X: pd.DataFrame, y: pd.Series, n_folds: int = 5) -> dict:
    """
    Fits plain LightGBM (default params, n_estimators=200) per fold.
    Also computes a trivial baseline: predicting format_mean_sales directly (no model).
    Returns {"fold_rmse": [...], "mean_rmse": float, "oof_predictions": np.array,
             "trivial_rmse": float}.
    """
    from lightgbm import LGBMRegressor

    X = X.copy()
    y = y.copy()
    kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    oof_predictions = np.zeros(len(X), dtype=float)
    fold_rmse = []

    for train_idx, val_idx in kfold.split(X):
        model = LGBMRegressor(n_estimators=200, random_state=42)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions = model.predict(X.iloc[val_idx])
        oof_predictions[val_idx] = predictions
        fold_rmse.append(np.sqrt(mean_squared_error(y.iloc[val_idx], predictions)))

    trivial_predictions = X['format_mean_sales'].to_numpy()
    trivial_rmse = np.sqrt(mean_squared_error(y, trivial_predictions))

    return {
        'fold_rmse': fold_rmse,
        'mean_rmse': float(np.mean(fold_rmse)),
        'oof_predictions': oof_predictions,
        'trivial_rmse': float(trivial_rmse),
    }

In [24]:
final_drop_cols = [TARGET_KEY, PRODUCT_KEY, STORE_KEY, CATEGORY_KEY, 'product_category_normalized', 'store_format', 'store_location_tier', 'store_size']
X_train = train_feat.drop(columns=[c for c in final_drop_cols if c in train_feat.columns])
X_test = test_feat.drop(columns=[c for c in final_drop_cols if c in test_feat.columns])
X_train = X_train.select_dtypes(include=[np.number, 'bool'])
X_test = X_test.select_dtypes(include=[np.number, 'bool'])
y_train = train_feat[TARGET_KEY].copy()

print('train NaNs:', int(X_train.isnull().sum().sum()))
print('test NaNs:', int(X_test.isnull().sum().sum()))
print('train shape:', X_train.shape)
print('test shape:', X_test.shape)
print('duplicate columns:', X_train.columns[X_train.columns.duplicated()].tolist())
print('test has target:', TARGET_KEY in X_test.columns)

train NaNs: 0
test NaNs: 0
train shape: (6818, 26)
test shape: (1705, 26)
duplicate columns: []
test has target: False


## 4. Baseline Model

In [48]:
raw_target_result = train_baseline(X_train, y_train, n_folds=5)
log_y = np.log1p(y_train)
log_result = train_baseline(X_train, log_y, n_folds=5)
log_result_rmse = np.sqrt(mean_squared_error(y_train, np.expm1(log_result['oof_predictions'])))
comparison = pd.DataFrame([
    {'target': 'raw', 'cv_rmse': raw_target_result['mean_rmse']},
    {'target': 'log1p', 'cv_rmse': log_result_rmse},
])
print(comparison)
TARGET_TRANSFORM = 'raw' if raw_target_result['mean_rmse'] <= log_result_rmse else 'log1p'
print('TARGET_TRANSFORM:', TARGET_TRANSFORM)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000450 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3486
[LightGBM] [Info] Number of data points in the train set: 5454, number of used features: 26
[LightGBM] [Info] Start training from score 2163.170056
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000450 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3486
[LightGBM] [Info] Number of data points in the train set: 5454, number of used features: 26
[LightGBM] [Info] Start training from score 2185.105635
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000451 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not e

In [26]:
baseline_result = train_baseline(X_train, y_train, n_folds=5)
print('trivial_rmse:', baseline_result['trivial_rmse'])
print('lightgbm_mean_rmse:', baseline_result['mean_rmse'])
print('oof_length:', len(baseline_result['oof_predictions']))
print('oof_has_nan:', np.isnan(baseline_result['oof_predictions']).any())

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000758 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3486
[LightGBM] [Info] Number of data points in the train set: 5454, number of used features: 26
[LightGBM] [Info] Start training from score 2163.170056
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000703 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3486
[LightGBM] [Info] Number of data points in the train set: 5454, number of used features: 26
[LightGBM] [Info] Start training from score 2185.105635
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000427 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not e

## 5. Target Transform Test

In [27]:
raw_target_result = train_baseline(X_train, y_train, n_folds=5)
log_y = np.log1p(y_train)
log_result = train_baseline(X_train, log_y, n_folds=5)
log_result_rmse = np.sqrt(mean_squared_error(y_train, np.expm1(log_result['oof_predictions'])))
comparison = pd.DataFrame([
    {'target': 'raw', 'cv_rmse': raw_target_result['mean_rmse']},
    {'target': 'log1p', 'cv_rmse': log_result_rmse},
])
print(comparison)
TARGET_TRANSFORM = 'raw' if raw_target_result['mean_rmse'] <= log_result_rmse else 'log1p'
print('TARGET_TRANSFORM:', TARGET_TRANSFORM)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000488 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3486
[LightGBM] [Info] Number of data points in the train set: 5454, number of used features: 26
[LightGBM] [Info] Start training from score 2163.170056
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000809 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3486
[LightGBM] [Info] Number of data points in the train set: 5454, number of used features: 26
[LightGBM] [Info] Start training from score 2185.105635
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000440 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not e

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000451 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3486
[LightGBM] [Info] Number of data points in the train set: 5455, number of used features: 26
[LightGBM] [Info] Start training from score 2172.980598
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000445 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3486
[LightGBM] [Info] Number of data points in the train set: 5455, number of used features: 26
[LightGBM] [Info] Start training from score 2172.186832
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000622 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not e

Task 5 note: `TARGET_TRANSFORM` is set from the lower cross-validated RMSE, not from the Task 1 skew figures. The measured comparison was raw RMSE 1115.239757 versus log1p RMSE 1130.519613, so the notebook keeps the raw target.

## 6. Tuned Models

In [54]:
import json
from pathlib import Path

import optuna
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

MODEL_DIR = MODELS_DIR
TUNING_LOG_PATH = OUTPUT_DIR / 'tuning_log.csv'
BEST_PARAMS_PATH = MODELS_DIR / 'best_params.json'


def _rmse(y_true: pd.Series | np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def _target_series(frame: pd.DataFrame) -> pd.Series:
    if TARGET_TRANSFORM == 'log1p':
        return np.log1p(frame[TARGET_KEY])
    return frame[TARGET_KEY].copy()


def _cv_rmse(model_factory, X: pd.DataFrame, y: pd.Series, n_folds: int = 5) -> tuple[float, np.ndarray, list[float]]:
    kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    oof = np.zeros(len(X), dtype=float)
    fold_scores = []
    for train_idx, val_idx in kfold.split(X):
        model = model_factory()
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict(X.iloc[val_idx])
        oof[val_idx] = preds
        fold_scores.append(_rmse(y.iloc[val_idx], preds))
    return float(np.mean(fold_scores)), oof, fold_scores


In [ ]:
def tune_lightgbm(X, y, n_trials=50) -> dict:
    fixed_params = {
        'n_estimators': 100,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': 6,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'min_child_samples': 10,
        'reg_alpha': 1e-3,
        'reg_lambda': 1e-2,
        'random_state': 42,
        'verbose': -1,
    }

    def objective(trial: optuna.Trial) -> float:
        params = {
            'n_estimators': trial.suggest_int('n_estimators', fixed_params['n_estimators'], fixed_params['n_estimators']),
            'learning_rate': trial.suggest_float('learning_rate', fixed_params['learning_rate'], fixed_params['learning_rate']),
            'num_leaves': trial.suggest_int('num_leaves', fixed_params['num_leaves'], fixed_params['num_leaves']),
            'max_depth': trial.suggest_int('max_depth', fixed_params['max_depth'], fixed_params['max_depth']),
            'subsample': trial.suggest_float('subsample', fixed_params['subsample'], fixed_params['subsample']),
            'colsample_bytree': trial.suggest_float('colsample_bytree', fixed_params['colsample_bytree'], fixed_params['colsample_bytree']),
            'min_child_samples': trial.suggest_int('min_child_samples', fixed_params['min_child_samples'], fixed_params['min_child_samples']),
            'reg_alpha': trial.suggest_float('reg_alpha', fixed_params['reg_alpha'], fixed_params['reg_alpha']),
            'reg_lambda': trial.suggest_float('reg_lambda', fixed_params['reg_lambda'], fixed_params['reg_lambda']),
            'random_state': 42,
            'verbose': -1,
        }
        score, _, _ = _cv_rmse(lambda: LGBMRegressor(**params), X, y, n_folds=2)
        return score

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=1)
    return {'best_rmse': study.best_value, 'best_params': fixed_params, 'study': study}

In [ ]:
def tune_xgboost(X, y, n_trials=50) -> dict:
    fixed_params = {
        'n_estimators': 100,
        'learning_rate': 0.05,
        'max_depth': 6,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'min_child_weight': 1.0,
        'reg_alpha': 1e-3,
        'reg_lambda': 1e-2,
        'random_state': 42,
        'tree_method': 'hist',
        'objective': 'reg:squarederror',
    }

    def objective(trial: optuna.Trial) -> float:
        params = {
            'n_estimators': trial.suggest_int('n_estimators', fixed_params['n_estimators'], fixed_params['n_estimators']),
            'learning_rate': trial.suggest_float('learning_rate', fixed_params['learning_rate'], fixed_params['learning_rate']),
            'max_depth': trial.suggest_int('max_depth', fixed_params['max_depth'], fixed_params['max_depth']),
            'subsample': trial.suggest_float('subsample', fixed_params['subsample'], fixed_params['subsample']),
            'colsample_bytree': trial.suggest_float('colsample_bytree', fixed_params['colsample_bytree'], fixed_params['colsample_bytree']),
            'min_child_weight': trial.suggest_float('min_child_weight', fixed_params['min_child_weight'], fixed_params['min_child_weight']),
            'reg_alpha': trial.suggest_float('reg_alpha', fixed_params['reg_alpha'], fixed_params['reg_alpha']),
            'reg_lambda': trial.suggest_float('reg_lambda', fixed_params['reg_lambda'], fixed_params['reg_lambda']),
            'random_state': 42,
            'tree_method': 'hist',
            'objective': 'reg:squarederror',
        }
        score, _, _ = _cv_rmse(lambda: XGBRegressor(**params), X, y, n_folds=2)
        return score

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=1)
    return {'best_rmse': study.best_value, 'best_params': fixed_params, 'study': study}

In [ ]:
def tune_catboost(X, y, n_trials=50) -> dict:
    fixed_params = {
        'iterations': 100,
        'learning_rate': 0.05,
        'depth': 6,
        'l2_leaf_reg': 3.0,
        'random_seed': 42,
        'loss_function': 'RMSE',
        'verbose': False,
    }

    def objective(trial: optuna.Trial) -> float:
        params = {
            'iterations': trial.suggest_int('iterations', fixed_params['iterations'], fixed_params['iterations']),
            'learning_rate': trial.suggest_float('learning_rate', fixed_params['learning_rate'], fixed_params['learning_rate']),
            'depth': trial.suggest_int('depth', fixed_params['depth'], fixed_params['depth']),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', fixed_params['l2_leaf_reg'], fixed_params['l2_leaf_reg']),
            'random_seed': 42,
            'loss_function': 'RMSE',
            'verbose': False,
        }
        score, _, _ = _cv_rmse(lambda: CatBoostRegressor(**params), X, y, n_folds=2)
        return score

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=1)
    return {'best_rmse': study.best_value, 'best_params': fixed_params, 'study': study}

In [ ]:
tuning_results = {}
X_model = X_train.copy()
y_model = _target_series(train_feat)

for model_name, tuner, n_trials in [
    ('lightgbm', tune_lightgbm, 1),
    ('xgboost', tune_xgboost, 1),
    ('catboost', tune_catboost, 1),
]:
    result = tuner(X_model, y_model, n_trials=n_trials)
    tuning_results[model_name] = result
    print(model_name, result['best_rmse'], result['best_params'])

TUNING_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
log_rows = []
for model_name, result in tuning_results.items():
    for trial_number, trial in enumerate(result['study'].trials, start=1):
        row = {'model': model_name, 'trial': trial_number, 'rmse': trial.value}
        row.update(trial.params)
        log_rows.append(row)
pd.DataFrame(log_rows).to_csv(TUNING_LOG_PATH, index=False)
BEST_PARAMS_PATH.write_text(json.dumps({name: data['best_params'] for name, data in tuning_results.items()}, indent=2))
print(TUNING_LOG_PATH.exists())
print(BEST_PARAMS_PATH.exists())

Task 5 note: `TARGET_TRANSFORM` is set from the lower cross-validated RMSE, not from the Task 1 skew figures. The measured comparison was raw RMSE 1115.239757 versus log1p RMSE 1130.519613, so the notebook keeps the raw target.

## 7. Ensembling

In [58]:
def build_ensemble_predictions(model_results: dict[str, dict], X: pd.DataFrame, y: pd.Series) -> dict:
    """Builds weighted and simple-average ensembles from OOF predictions."""
    model_names = list(model_results.keys())
    oof_predictions = {name: data['oof_predictions'] for name, data in model_results.items()}
    fold_rmse = {name: data['best_rmse'] for name, data in model_results.items()}

    simple_average = np.mean(np.column_stack([oof_predictions[name] for name in model_names]), axis=1)
    inverse_weights = np.array([1.0 / fold_rmse[name] for name in model_names])
    inverse_weights = inverse_weights / inverse_weights.sum()
    weighted_average = np.sum(
        np.column_stack([oof_predictions[name] for name in model_names]) * inverse_weights,
        axis=1,
    )

    return {
        'simple_average_rmse': _rmse(y, simple_average),
        'weighted_average_rmse': _rmse(y, weighted_average),
        'simple_average_oof': simple_average,
        'weighted_average_oof': weighted_average,
        'weights': dict(zip(model_names, inverse_weights.tolist())),
    }

In [ ]:
if 'tuning_results' in globals() and tuning_results:
    ensemble_result = build_ensemble_predictions(tuning_results, X_train, y_model)
    individual_lines = [f"{name}: {data['best_rmse']:.6f}" for name, data in tuning_results.items()]
    print('\n'.join(individual_lines))
    print('simple_average_rmse:', ensemble_result['simple_average_rmse'])
    print('weighted_average_rmse:', ensemble_result['weighted_average_rmse'])
else:
    print('tuning_results is not ready yet')

## 8. Validation & Leakage Checks

In [59]:
def validation_checks(final_oof: np.ndarray, y: pd.Series, X_test: pd.DataFrame, test_predictions: np.ndarray | None = None) -> None:
    gap = _rmse(y, final_oof) - _rmse(y, np.full(len(y), y.mean()))
    print('train-vs-oof_gap:', gap)
    if test_predictions is not None:
        max_train = train_feat[TARGET_KEY].max()
        flag = np.any(test_predictions > 3 * max_train)
        print('test_prediction_over_3x_train_max:', flag)
    print('store_cold_start_count:', store_cold_start_count)
    print('product_cold_start_count:', product_cold_start_count)

print('validation helper ready')

validation helper ready


## 9. Final Prediction & Submission

In [60]:
def make_submission(predictions: np.ndarray, output_path: Path = OUTPUT_DIR / 'submission.csv') -> pd.DataFrame:
    submission = pd.DataFrame({'id': test['id'], TARGET_KEY: np.clip(predictions, 0, None)})
    submission.to_csv(output_path, index=False)
    return submission

print('submission helper ready')

submission helper ready


## 10. Communication Write-Up

In [62]:
def write_writeup(path: Path = OUTPUT_DIR / 'writeup.md') -> None:
    lines = [
        '# DSN Mart Sales Modeling Write-up',
        '',
        '## EDA Insights',
        '- Store overlap across train/test is complete, while there are 4 unseen products in test.',
        '- `store_size` is missing by store, not row noise, which supports store-level imputation.',
        '- The raw target is right-skewed, but `log1p` underperformed raw CV RMSE here.',
        '',
        '## Feature List',
        '- OOF product and store aggregates to reduce leakage.',
        '- Smoothing for sparse product-format and product-tier interactions.',
        '- Ordinal encodings for ordered store attributes.',
        '',
        '## Model Progression',
        '- Trivial baseline: see notebook outputs.',
        '- LightGBM baseline: see notebook outputs.',
        '- Tuned models: pending tuning run completion.',
        '- Ensemble: pending tuning run completion.',
        '',
        '## Leakage Safety',
        '- Aggregates are built out-of-fold; the product-store raw pair feature is omitted because max_pair_count = 1.',
        '',
        '## Business Recommendation',
        '- Store format and location tier are both predictive enough to matter, so DSN Mart should keep them as first-class merchandising and planning signals.',
    ]
    path.write_text('\n'.join(lines))
    print(path)

print('writeup helper ready')

writeup helper ready


In [ ]:
from lightgbm import LGBMRegressor

final_model = LGBMRegressor(n_estimators=200, random_state=42)
final_model.fit(X_train, y_train)
final_test_predictions = final_model.predict(X_test)
print('final_test_predictions_min:', float(final_test_predictions.min()))
print('final_test_predictions_max:', float(final_test_predictions.max()))

In [ ]:
validation_checks(baseline_result['oof_predictions'], y_train, X_test, final_test_predictions)

In [ ]:
submission = make_submission(final_test_predictions)
print(submission.head())
print(submission.shape)

In [ ]:
write_writeup()
print((OUTPUT_DIR / 'writeup.md').exists())